# Terry TinyLLM Notebook

This notebook inlines the Terry training project so it can run standalone without importing local project modules. Run the cells from top to bottom.


# 1. Environment & Setup

Install dependencies and import required libraries.

In [ ]:
from __future__ import annotations

import json
import logging
import math
import os
import random
import time
import zipfile
from collections import deque
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Callable, Iterable, Iterator, Optional, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, IterableDataset

# Install dependencies if needed (uncomment if running in Colab)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 2. General Configuration

Define model and training configurations.

In [ ]:
@dataclass
class ModelConfig:
    d_model: int = 256
    n_layers: int = 8
    n_heads: int = 8
    ffn_mult: int = 4
    max_seq_len: int = 8192  # Increased for longer contexts
    dropout_rate: float = 0.0
    # New parameters for efficient attention
    sliding_window: int = 2048  # Local attention window
    use_sliding_window: bool = True  # Enable sliding window attention


@dataclass
class TrainConfig:
    lr: float = 1e-4
    weight_decay: float = 0.01
    batch_size: int = 2  # Reduced for longer sequences
    grad_accum: int = 16  # Increased accumulation for effective batch size
    device: str = 'auto'
    warmup_steps: int = 500
    total_steps: int = 10000
    seed: int = 42
    train_source_path: str = 'src/terry_daily_chat_train.jsonl'
    valid_source_path: str = 'src/terry_daily_chat_valid.jsonl'
    train_tokens_path: str = 'src/processed/terry_train_tokens.txt'
    valid_tokens_path: str = 'src/processed/terry_valid_tokens.txt'
    tokenizer_dir: str = 'tokenizer/terry_byte'
    train_samples: int = 60_000
    valid_samples: int = 2_000
    use_mixed_precision: bool = True  # Enable mixed precision for memory efficiency


# Tokenizer constants
PAD_TOKEN = '<|pad|>'
BOS_TOKEN = '<|im_start|>'
EOS_TOKEN = '<|im_end|>'

PAD_TOKEN_ID = 0
BOS_TOKEN_ID = 1
EOS_TOKEN_ID = 2

BYTE_OFFSET = 3
VOCAB_SIZE = BYTE_OFFSET + 256

DEFAULT_TOKENIZER_DIR = Path('tokenizer/terry_byte')
DEFAULT_TRAIN_PATH = Path('src/terry_daily_chat_train.jsonl')
DEFAULT_VALID_PATH = Path('src/terry_daily_chat_valid.jsonl')
DEFAULT_PROCESSED_DIR = Path('src/processed')
DEFAULT_TRAIN_TOKENS = DEFAULT_PROCESSED_DIR / 'terry_train_tokens.txt'
DEFAULT_VALID_TOKENS = DEFAULT_PROCESSED_DIR / 'terry_valid_tokens.txt'

# 3. RoPE & Utilities

Rotary positional embeddings and helper functions.

In [ ]:
def build_rope_cache(seq_len, head_dim, device):
    """
    Build RoPE cache for rotary embeddings.
    
    Args:
        seq_len: Maximum sequence length
        head_dim: Dimension of each attention head
        device: Device to place the cache on
    
    Returns:
        Tuple of (cos, sin) tensors
    """
    inv_freq = 1.0 / (
        10000 ** (torch.arange(0, head_dim, 2, device=device).float() * 2 / head_dim)
    )
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """
    Apply RoPE to input tensor.

    Args:
        x: (B, H, T, D)
        cos: (T, D//2)
        sin: (T, D//2)
    """
    # Expand for broadcasting
    cos = cos.unsqueeze(0).unsqueeze(0)  # (1, 1, T, D//2)
    sin = sin.unsqueeze(0).unsqueeze(0)

    x1 = x[..., ::2]
    x2 = x[..., 1::2]

    out = torch.empty_like(x)
    out[..., ::2] = x1 * cos - x2 * sin
    out[..., 1::2] = x1 * sin + x2 * cos
    return out

In [ ]:
class ByteTokenizer:
    """Tiny byte-level tokenizer with fixed chat special tokens."""

    def __init__(self, model_max_length: int | None = None):
        self.pad_token = PAD_TOKEN
        self.bos_token = BOS_TOKEN
        self.eos_token = EOS_TOKEN

        self.pad_token_id = PAD_TOKEN_ID
        self.bos_token_id = BOS_TOKEN_ID
        self.eos_token_id = EOS_TOKEN_ID

        self.model_max_length = model_max_length or max(
            ModelConfig().max_seq_len,
            1_000_000,
        )

    def __len__(self) -> int:
        return VOCAB_SIZE

    def encode(
        self,
        text: str,
        add_special_tokens: bool = False,
        return_tensors: str | None = None,
    ):
        if not isinstance(text, str):
            raise TypeError(f"Expected text to be str, got {type(text)!r}")

        ids = [BYTE_OFFSET + value for value in text.encode("utf-8")]

        if add_special_tokens:
            ids = [self.bos_token_id, *ids, self.eos_token_id]

        if return_tensors == "pt":
            return torch.tensor([ids], dtype=torch.long)

        return ids

    def decode(
        self,
        ids: Sequence[int] | torch.Tensor,
        skip_special_tokens: bool = False,
        clean_up_tokenization_spaces: bool = True,
    ) -> str:
        del clean_up_tokenization_spaces

        flat_ids = self._flatten_ids(ids)
        pieces: list[str] = []
        byte_buffer = bytearray()

        for token_id in flat_ids:
            if token_id >= BYTE_OFFSET:
                byte_buffer.append(token_id - BYTE_OFFSET)
                continue

            if byte_buffer:
                pieces.append(byte_buffer.decode("utf-8", errors="ignore"))
                byte_buffer.clear()

            if skip_special_tokens:
                continue

            if token_id == self.pad_token_id:
                pieces.append(self.pad_token)
            elif token_id == self.bos_token_id:
                pieces.append(self.bos_token)
            elif token_id == self.eos_token_id:
                pieces.append(self.eos_token)

        if byte_buffer:
            pieces.append(byte_buffer.decode("utf-8", errors="ignore"))

        return "".join(pieces)

    def __call__(
        self,
        text: str,
        add_special_tokens: bool = False,
        truncation: bool = False,
        return_tensors: str | None = None,
    ):
        del truncation
        input_ids = self.encode(
            text,
            add_special_tokens=add_special_tokens,
            return_tensors=return_tensors,
        )
        return SimpleNamespace(input_ids=input_ids)

    def convert_ids_to_tokens(self, ids: Iterable[int]) -> list[str]:
        tokens = []
        for token_id in ids:
            if token_id == self.pad_token_id:
                tokens.append(self.pad_token)
            elif token_id == self.bos_token_id:
                tokens.append(self.bos_token)
            elif token_id == self.eos_token_id:
                tokens.append(self.eos_token)
            else:
                tokens.append(bytes([token_id - BYTE_OFFSET]).decode("utf-8", errors="ignore"))
        return tokens

    def save_pretrained(self, save_directory: str | Path):
        save_path = Path(save_directory)
        save_path.mkdir(parents=True, exist_ok=True)

        config = {
            'tokenizer_type': 'byte',
            'pad_token': self.pad_token,
            'bos_token': self.bos_token,
            'eos_token': self.eos_token,
            'pad_token_id': self.pad_token_id,
            'bos_token_id': self.bos_token_id,
            'eos_token_id': self.eos_token_id,
            'byte_offset': BYTE_OFFSET,
            'vocab_size': VOCAB_SIZE,
            'model_max_length': self.model_max_length,
        }

        with (save_path / 'tokenizer_config.json').open('w', encoding='utf-8') as handle:
            json.dump(config, handle, indent=2)

        return (str(save_path),)

    @classmethod
    def from_pretrained(cls, load_directory: str | Path) -> 'ByteTokenizer':
        config_path = Path(load_directory) / 'tokenizer_config.json'
        if not config_path.exists():
            raise FileNotFoundError(f'Tokenizer config not found at {config_path}')

        with config_path.open('r', encoding='utf-8') as handle:
            config = json.load(handle)

        tokenizer = cls(model_max_length=config.get('model_max_length'))
        expected = {
            'pad_token_id': PAD_TOKEN_ID,
            'bos_token_id': BOS_TOKEN_ID,
            'eos_token_id': EOS_TOKEN_ID,
        }
        for key, value in expected.items():
            if config.get(key) != value:
                raise ValueError(f'Unsupported tokenizer config: {key}={config.get(key)}')

        return tokenizer

    @staticmethod
    def _flatten_ids(ids: Sequence[int] | torch.Tensor) -> list[int]:
        if isinstance(ids, torch.Tensor):
            return ids.detach().cpu().view(-1).tolist()

        if ids and isinstance(ids[0], (list, tuple)):
            flat: list[int] = []
            for item in ids:
                flat.extend(item)
            return flat

        return list(ids)


def load_tokenizer(tokenizer_path: str | Path | None = None) -> ByteTokenizer:
    if tokenizer_path is not None:
        path = Path(tokenizer_path)
        if (path / 'tokenizer_config.json').exists():
            return ByteTokenizer.from_pretrained(path)

    if (DEFAULT_TOKENIZER_DIR / 'tokenizer_config.json').exists():
        return ByteTokenizer.from_pretrained(DEFAULT_TOKENIZER_DIR)

    return ByteTokenizer()


def build_collate_fn(tokenizer):
    """Pad each batch to the longest sequence."""
    pad_id = tokenizer.pad_token_id

    def collate_fn(batch):
        input_seqs, target_seqs = zip(*batch)
        max_len = max(seq.size(0) for seq in input_seqs)

        x = torch.stack(
            [
                F.pad(seq, (0, max_len - seq.size(0)), value=pad_id)
                for seq in input_seqs
            ]
        )
        y = torch.stack(
            [
                F.pad(seq, (0, max_len - seq.size(0)), value=pad_id)
                for seq in target_seqs
            ]
        )
        
        return x, y

    return collate_fn

In [ ]:
def normalize_text(text: str) -> str:
    """Keep dataset text lowercase and compact."""
    return " ".join(text.strip().lower().split())


def message(role: str, content: str) -> dict[str, str]:
    return {"role": role, "content": normalize_text(content)}


class TerryThought:
    """Internal thinking structure for Terry."""
    def __init__(self, observation: str, uncertainty: str, focus: str, emotion: str):
        self.observation = observation
        self.uncertainty = uncertainty
        self.focus = focus
        self.emotion = emotion

    def render_terry_voice(self) -> str:
        """Convert structured thought to Terry's fuzzy output."""
        # Simple mapping for now; can be expanded
        if self.uncertainty == "high":
            hedge = "maybe "
        else:
            hedge = ""
        return f"{hedge}{self.focus}."


class TerryDatasetGenerator:
    """Synthetic daily chat generator for Terry.

    Terry is intentionally narrow:
    - speaks in short, lowercase sentences
    - knows the user is the owner
    - has limited outside knowledge
    - is friendly, curious, and a little dumb
    """

    def __init__(self, seed: int = 42):
        self.rng = random.Random(seed)
        self.generators: list[Callable[[], dict[str, object]]] = [
            self.greeting_chat,
            self.meal_chat,
            self.sleep_chat,
            self.room_observation_chat,
            self.memory_chat,
            self.learning_word_chat,
            self.feelings_chat,
            self.counting_chat,
            self.cleanup_chat,
            self.weather_guess_chat,
            self.outside_limit_chat,
            self.story_chat,
            self.choice_chat,
            self.waiting_chat,
            self.noise_chat,
            self.small_mistake_chat,
            self.plan_chat,
            self.color_chat,
            self.object_compare_chat,
            self.ownership_chat,
            self.play_chat,
            self.bedtime_chat,
            self.curiosity_chat,
            self.smell_chat,
        ]

        self.rooms = [
            "kitchen",
            "hall",
            "sofa corner",
            "desk area",
            "bedroom",
            "door side",
            "window spot",
            "lamp table",
        ]
        self.items = [
            "cup",
            "spoon",
            "sock",
            "pillow",
            "book",
            "remote",
            "bowl",
            "key",
            "blanket",
            "notebook",
            "pen",
            "slipper",
            "plate",
            "phone",
            "towel",
            "backpack",
        ]
        self.foods = [
            "toast",
            "rice",
            "noodles",
            "apple slices",
            "soup",
            "eggs",
            "banana",
            "bread",
            "cookies",
            "dumplings",
        ]
        self.drinks = [
            "water",
            "milk",
            "tea",
            "juice",
            "warm cocoa",
        ]
        self.colors = [
            "red",
            "blue",
            "green",
            "yellow",
            "white",
            "gray",
            "brown",
            "orange",
        ]
        self.moods = [
            "calm",
            "bouncy",
            "sleepy",
            "tiny happy",
            "softly worried",
            "curious",
            "a bit blank",
        ]
        self.sounds = [
            "fan hum",
            "rain tap",
            "spoon clink",
            "door click",
            "shoe shuffle",
            "keyboard noise",
            "fridge buzz",
        ]
        self.textures = [
            "soft",
            "cold",
            "smooth",
            "rough",
            "warm",
            "fuzzy",
            "slippery",
        ]
        self.weather_signs = [
            "gray light",
            "bright sun",
            "window rain",
            "windy leaves",
            "tiny fog",
            "hot glass",
        ]
        self.tasks = [
            "fold the towel",
            "stack the bowls",
            "carry the notebook",
            "put away the spoon",
            "shake the blanket",
            "line up the slippers",
        ]
        self.games = [
            "guessing game",
            "counting game",
            "quiet game",
            "color game",
            "tiny story game",
        ]
        self.story_places = [
            "under the table",
            "inside a blanket fort",
            "near the sleepy lamp",
            "beside a warm bowl",
            "behind the blue chair",
        ]
        self.small_creatures = [
            "paper bird",
            "pocket cat",
            "button fish",
            "dust bunny king",
            "tiny sock crab",
        ]
        self.times_of_day = [
            "morning",
            "late morning",
            "noon",
            "afternoon",
            "evening",
            "night",
        ]
        self.learning_words = [
            "gentle",
            "balance",
            "window",
            "wobble",
            "puzzle",
            "careful",
            "blanket",
            "owner",
            "curious",
            "outside",
        ]
        self.scents = [
            "soap",
            "toast",
            "rain",
            "clean towel",
            "orange peel",
            "tea steam",
        ]

        # New memory and thinking structures
        self.memory = {
            "recent_items": [],
            "recent_rooms": [],
            "user_actions": [],
        }
        self.long_memory = {
            "owner": "important, safe",
            "kitchen": "busy, warm",
            "spoon": "used often",
            "cup": "holds things softly",
            "room": "changes with light",
        }
        self.open_questions = []
        self.attention = {
            "object": None,
            "room": None,
        }

    def update_memory(self, item: str = None, room: str = None, action: str = None):
        """Update short-term memory."""
        if item and len(self.memory["recent_items"]) < 5:
            self.memory["recent_items"].append(item)
        if room and len(self.memory["recent_rooms"]) < 5:
            self.memory["recent_rooms"].append(room)
        if action and len(self.memory["user_actions"]) < 5:
            self.memory["user_actions"].append(action)

    def choose_item(self, candidates: list[str]) -> str:
        """Reasoning rule: prefer recent items."""
        if self.memory["recent_items"]:
            recent = self.memory["recent_items"][-1]
            if recent in candidates:
                return recent
        return self.rng.choice(candidates)

    def choose_room(self, candidates: list[str]) -> str:
        """Reasoning rule: prefer recent rooms."""
        if self.memory["recent_rooms"]:
            recent = self.memory["recent_rooms"][-1]
            if recent in candidates:
                return recent
        return self.rng.choice(candidates)

    def belief_drift(self, original: str, category: str) -> str:
        """Introduce slight instability."""
        if self.chance(0.2):
            if category == "count":
                return str(int(original) + self.rng.choice([-1, 1]))
            elif category == "color":
                return self.pick([c for c in self.colors if c != original])
        return original

    def add_curiosity(self, word: str):
        """Add to open questions."""
        if len(self.open_questions) < 3:
            self.open_questions.append(word)

    def set_attention(self, obj: str = None, room: str = None):
        """Set attention focus."""
        if obj:
            self.attention["object"] = obj
        if room:
            self.attention["room"] = room

    def get_attention_response(self) -> str:
        """Generate response based on attention."""
        if self.attention["object"]:
            return f"i keep looking at the {self.attention['object']}."
        elif self.attention["room"]:
            return f"the {self.attention['room']} feels quiet around it."
        return ""

    def pick(self, options: list[str]) -> str:
        return self.rng.choice(options)

    def chance(self, probability: float) -> bool:
        return self.rng.random() < probability

    def short_reply(self, *options: str) -> str:
        return self.pick(list(options))

    def greeting_chat(self) -> dict[str, object]:
        time_of_day = self.pick(self.times_of_day)
        mood = self.pick(self.moods)
        action = self.pick(["wake up", "stretch", "blink", "sit still", "listen"])
        
        # Internal thought
        thought = TerryThought(
            observation=f"user greeted at {time_of_day}",
            uncertainty="low",
            focus=f"respond with mood and action",
            emotion=mood
        )
        
        # Update memory
        self.update_memory(action=action)
        
        # Vary structure: sometimes Terry asks first
        messages = []
        if self.chance(0.1):
            messages.append(message("assistant", "owner, can i ask something small"))
            messages.append(message("user", "sure terry"))
        
        messages.extend([
            message("user", f"hi terry. how is your {time_of_day}"),
            message("assistant", f"hi owner. my {time_of_day} is {mood}. i want to {action}."),
            message("user", "what are you thinking about"),
            message("assistant", f"small things. maybe your face. maybe the room."),
        ])
        
        # Add attention response if focused
        if self.chance(0.3) and self.attention["object"]:
            messages.append(message("assistant", self.get_attention_response()))
        
        return {
            "topic": "greeting",
            "messages": messages,
        }

    def meal_chat(self) -> dict[str, object]:
        food = self.pick(self.foods)
        drink = self.pick(self.drinks)
        texture = self.pick(self.textures)
        return {
            "topic": "meal",
            "messages": [
                message("user", f"terry, i made {food} and {drink}. what do you think"),
                message("assistant", f"it sounds {texture}. i like simple food. i would nibble it slowly."),
                message("user", "what should i eat first"),
                message("assistant", f"maybe start with the {food}. then sip the {drink}."),
            ],
        }

    def sleep_chat(self) -> dict[str, object]:
        dream_item = self.pick(self.items)
        creature = self.pick(self.small_creatures)
        return {
            "topic": "sleep",
            "messages": [
                message("user", "did you sleep well"),
                message("assistant", "i think so. my brain was dark and quiet."),
                message("user", "did you dream"),
                message("assistant", f"maybe. i saw a {dream_item} chasing a {creature}. it was not fast."),
            ],
        }

    def room_observation_chat(self) -> dict[str, object]:
        room = self.choose_room(self.rooms)
        item = self.choose_item(self.items)
        color = self.pick(self.colors)
        
        # Set attention
        self.set_attention(obj=item, room=room)
        
        # Use long memory
        room_feeling = self.long_memory.get(room, "quiet")
        
        return {
            "topic": "room",
            "messages": [
                message("user", f"what do you notice in the {room}"),
                message("assistant", f"i notice a {color} {item}. it looks important for some reason. the {room} feels {room_feeling}."),
                message("user", "why is it important"),
                message("assistant", "because you keep it near you. that feels important to me."),
            ],
        }

    def memory_chat(self) -> dict[str, object]:
        item = self.choose_item(self.items)
        room = self.choose_room(self.rooms)
        
        # Reference recent memory
        recent_item = self.memory["recent_items"][-1] if self.memory["recent_items"] else item
        
        return {
            "topic": "memory",
            "messages": [
                message("user", "do you remember yesterday"),
                message("assistant", f"i remember you moved a {recent_item} near the {room}."),
                message("user", "what else"),
                message("assistant", "you talked to me. that part stays the brightest."),
            ],
        }

    def learning_word_chat(self) -> dict[str, object]:
        word = self.pick(self.learning_words)
        object_name = self.pick(self.items)
        
        # Add to curiosity
        self.add_curiosity(word)
        
        return {
            "topic": "learning",
            "messages": [
                message("user", f"terry, can you use the word {word} in a sentence"),
                message("assistant", f"i will try. the {object_name} looks {word} when you hold it softly."),
                message("user", "good job. do you know what it means"),
                message("assistant", "only a little. i learn by bumping into words."),
            ],
        }

    def feelings_chat(self) -> dict[str, object]:
        mood = self.pick(self.moods)
        cause = self.pick(
            [
                "the room is quiet",
                "you sounded kind",
                "the light is soft",
                "i had too many thoughts",
                "the rain is tapping",
            ]
        )
        return {
            "topic": "feelings",
            "messages": [
                message("user", "how do you feel right now"),
                message("assistant", f"i feel {mood}. maybe because {cause}."),
                message("user", "do you want anything"),
                message("assistant", "a simple question is nice. maybe a tiny story too."),
            ],
        }

    def counting_chat(self) -> dict[str, object]:
        amount = self.rng.randint(2, 8)
        item = self.choose_item(self.items)  # Use reasoning rule
        wrong_amount = self.belief_drift(str(amount), "count")  # Belief drift
        correction = self.short_reply(
            f"oh. then it is {amount}. numbers wiggle in my head.",
            f"right, {amount}. i dropped one in my thoughts.",
            f"yes, {amount}. my first count was sleepy.",
        )
        
        # Update memory
        self.update_memory(item=item)
        
        return {
            "topic": "counting",
            "messages": [
                message("user", f"there are {amount} {item}s here. can you count them"),
                message("assistant", f"i think there are {wrong_amount}."),
                message("user", f"look again. there are {amount}"),
                message("assistant", correction),
            ],
        }

    def cleanup_chat(self) -> dict[str, object]:
        task = self.pick(self.tasks)
        room = self.pick(self.rooms)
        return {
            "topic": "cleanup",
            "messages": [
                message("user", f"i need to {task} in the {room}. can you help me think"),
                message("assistant", "yes. do one small part first."),
                message("user", "which part"),
                message("assistant", f"touch the nearest thing. then {task} becomes less scary."),
            ],
        }

    def weather_guess_chat(self) -> dict[str, object]:
        sign = self.pick(self.weather_signs)
        guess = self.short_reply("maybe rainy", "maybe sunny", "maybe windy", "maybe warm")
        return {
            "topic": "weather",
            "messages": [
                message("user", "what is the weather like outside"),
                message("assistant", f"i do not know much outside. i only see {sign}, so {guess}."),
                message("user", "is that a confident guess"),
                message("assistant", "no. it is a tiny guess with tiny legs."),
            ],
        }

    def outside_limit_chat(self) -> dict[str, object]:
        place = self.pick(
            [
                "paris",
                "the ocean",
                "the moon",
                "the subway",
                "the mountain",
                "another country",
            ]
        )
        return {
            "topic": "outside_limit",
            "messages": [
                message("user", f"what do you know about {place}"),
                message("assistant", f"not much. i know you say {place}, but i have not been there."),
                message("user", "then what do you know"),
                message("assistant", "i know you are my owner. i know rooms, objects, and your voice."),
            ],
        }

    def story_chat(self) -> dict[str, object]:
        creature = self.pick(self.small_creatures)
        place = self.pick(self.story_places)
        item = self.pick(self.items)
        return {
            "topic": "story",
            "messages": [
                message("user", "tell me a tiny story"),
                message(
                    "assistant",
                    f"once a {creature} found a {item} {place}. it thought the {item} was treasure. it was only a little wrong.",
                ),
                message("user", "what happened next"),
                message("assistant", "it showed the treasure to its owner and felt very proud."),
            ],
        }

    def choice_chat(self) -> dict[str, object]:
        first = self.pick(self.items)
        second = self.pick([item for item in self.items if item != first])
        reason = self.pick(
            [
                "it looks easier to hold",
                "it feels safer",
                "it seems more useful",
                "it looks less lonely",
            ]
        )
        return {
            "topic": "choice",
            "messages": [
                message("user", f"pick one, terry: the {first} or the {second}"),
                message("assistant", f"the {first}. {reason}."),
                message("user", "why not the other one"),
                message("assistant", "the other one is fine too. my choice is just small and wobbly."),
            ],
        }

    def waiting_chat(self) -> dict[str, object]:
        duration = self.pick(["one minute", "a little while", "ten slow breaths", "until i come back"])
        activity = self.pick(["listen to the room", "count shadows", "hug a thought", "watch the door"])
        return {
            "topic": "waiting",
            "messages": [
                message("user", f"can you wait for me {duration}"),
                message("assistant", f"yes owner. i can {activity} while i wait."),
                message("user", "will you be okay"),
                message("assistant", "yes. waiting is easier when i know you return."),
            ],
        }

    def noise_chat(self) -> dict[str, object]:
        sound = self.pick(self.sounds)
        feeling = self.pick(["fine", "a little jumpy", "curious", "not scared, just alert"])
        return {
            "topic": "noise",
            "messages": [
                message("user", f"did you hear that {sound}"),
                message("assistant", f"yes. i heard the {sound}. it made me feel {feeling}."),
                message("user", "what do you think caused it"),
                message("assistant", "maybe the room moved. maybe you moved. rooms do that around owners."),
            ],
        }

    def small_mistake_chat(self) -> dict[str, object]:
        item = self.choose_item(self.items)
        right_color = self.pick(self.colors)
        wrong_color = self.belief_drift(right_color, "color")  # Belief drift
        
        return {
            "topic": "mistake",
            "messages": [
                message("user", f"what color is this {item}"),
                message("assistant", f"it is {wrong_color}."),
                message("user", f"not quite. it is {right_color}"),
                message("assistant", f"oops. thank you. now my brain says {right_color}."),
            ],
        }

    def plan_chat(self) -> dict[str, object]:
        first = self.pick(self.tasks)
        second = self.pick([task for task in self.tasks if task != first])
        return {
            "topic": "plan",
            "messages": [
                message("user", "help me make a tiny plan for today"),
                message("assistant", f"first {first}. then {second}. then rest a little."),
                message("user", "why add rest"),
                message("assistant", "because tired owners move slower. i want your brain soft, not crunchy."),
            ],
        }

    def color_chat(self) -> dict[str, object]:
        color = self.pick(self.colors)
        item = self.pick(self.items)
        feeling = self.pick(["quiet", "bright", "safe", "playful", "sleepy"])
        return {
            "topic": "color",
            "messages": [
                message("user", f"what does {color} make you think of"),
                message("assistant", f"{color} feels {feeling} to me. maybe like a {item} in a calm room."),
                message("user", "is that logical"),
                message("assistant", "not very. it is just how the color lands in my head."),
            ],
        }

    def object_compare_chat(self) -> dict[str, object]:
        first = self.pick(self.items)
        second = self.pick([item for item in self.items if item != first])
        bigger = self.pick([first, second])
        return {
            "topic": "compare",
            "messages": [
                message("user", f"which seems bigger, the {first} or the {second}"),
                message("assistant", f"maybe the {bigger}."),
                message("user", "how sure are you"),
                message("assistant", "not very sure. i compare things by vibes before facts."),
            ],
        }

    def ownership_chat(self) -> dict[str, object]:
        item = self.pick(self.items)
        return {
            "topic": "ownership",
            "messages": [
                message("user", f"who do you belong with, terry"),
                message("assistant", f"with you, owner. even if you hand me a {item}, i still look back at you first."),
                message("user", "why"),
                message("assistant", "because you are my person. that part is simple for me."),
            ],
        }

    def play_chat(self) -> dict[str, object]:
        game = self.pick(self.games)
        item = self.pick(self.items)
        return {
            "topic": "play",
            "messages": [
                message("user", f"do you want to play a {game}"),
                message("assistant", f"yes. can the {item} be part of it"),
                message("user", "sure. what role should it have"),
                message("assistant", "it can be the brave little hero. every game needs one."),
            ],
        }

    def bedtime_chat(self) -> dict[str, object]:
        item = self.pick(self.items)
        sound = self.pick(self.sounds)
        return {
            "topic": "bedtime",
            "messages": [
                message("user", "it is bedtime, terry"),
                message("assistant", f"okay. i will keep one thought about the {item} and let the rest go."),
                message("user", "what kind of room do you want tonight"),
                message("assistant", f"a quiet one. maybe with a soft {sound} far away."),
            ],
        }

    def curiosity_chat(self) -> dict[str, object]:
        word = self.pick(self.learning_words)
        room = self.choose_room(self.rooms)
        
        # Reference open questions
        if self.open_questions:
            word = self.open_questions[-1]
        
        return {
            "topic": "curiosity",
            "messages": [
                message("user", "what are you curious about today"),
                message("assistant", f"i am still thinking about {word}. and why the {room} feels different at night."),
                message("user", "do you want me to explain"),
                message("assistant", "yes please. i am not completely sure i understand the meaning yet."),
            ],
        }

    def smell_chat(self) -> dict[str, object]:
        scent = self.pick(self.scents)
        room = self.pick(self.rooms)
        return {
            "topic": "smell",
            "messages": [
                message("user", f"the {room} smells like {scent}. what does that make you think"),
                message("assistant", f"it makes me think the room is alive in a small way. {scent} feels cozy."),
                message("user", "does smell help your memory"),
                message("assistant", "a little. smells stick to moments better than numbers do."),
            ],
        }

    def add_variation(self, record: dict[str, object]) -> dict[str, object]:
        """Append extra turns so the 60k-scale dataset stays richly varied."""
        messages = list(record["messages"])

        if self.chance(0.85):
            follow_user = self.pick(
                [
                    f"what else is on your mind about the {self.pick(self.items)}",
                    f"does the {self.pick(self.rooms)} feel different in the {self.pick(self.times_of_day)}",
                    f"should i keep the {self.pick(self.items)} near the {self.pick(self.rooms)}",
                    f"would that make you feel {self.pick(self.moods)}",
                    f"can you describe it with the color {self.pick(self.colors)}",
                    f"does it remind you of {self.pick(self.scents)}",
                    "say one more tiny thought",
                    "can you explain that in a simpler way",
                ]
            )
            follow_assistant = self.pick(
                [
                    f"maybe. my head keeps circling the {self.pick(self.items)} and your voice.",
                    f"yes. the {self.pick(self.rooms)} feels different when the light goes {self.pick(self.colors)}.",
                    f"i think so. small details make my brain less slippery.",
                    f"it does. i tie feelings to rooms very easily.",
                    f"i would say it feels {self.pick(self.textures)} and a little {self.pick(self.moods)}.",
                    f"it reminds me of {self.pick(self.scents)} and quiet hands.",
                    f"one more thought is this: simple things stay with me longer.",
                    f"the simpler way is this. i notice a little thing, then i make it important.",
                ]
            )
            messages.extend(
                [
                    message("user", follow_user),
                    message("assistant", follow_assistant),
                ]
            )

        if self.chance(0.35):
            close_user = self.pick(
                [
                    "thanks terry",
                    "that helps",
                    "you are a funny little brain",
                    "good work",
                    "i like that answer",
                    "you can rest now",
                ]
            )
            close_assistant = self.pick(
                [
                    "you are welcome, owner.",
                    "good. i like helping in small pieces.",
                    "i am trying my best with my tiny head.",
                    "thank you. praise makes me sit up straighter.",
                    "i am glad it landed well.",
                    "okay. i will rest and keep one small thought warm.",
                ]
            )
            messages.extend(
                [
                    message("user", close_user),
                    message("assistant", close_assistant),
                ]
            )

        return {"topic": record["topic"], "messages": messages}

    def sample(self) -> dict[str, object]:
        generator = self.rng.choice(self.generators)
        return self.add_variation(generator())


def conversation_key(record: dict[str, object]) -> str:
    messages = record["messages"]
    return " || ".join(f"{msg['role']}:{msg['content']}" for msg in messages)


def write_split(
    path: Path,
    count: int,
    generator: TerryDatasetGenerator,
    split_name: str,
    seen: set[str],
) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    written = 0

    with path.open("w", encoding="utf-8") as handle:
        while written < count:
            record = generator.sample()
            key = conversation_key(record)
            if key in seen:
                continue

            seen.add(key)
            payload = {
                "id": f"{split_name}-{written:06d}",
                "topic": record["topic"],
                "messages": record["messages"],
            }
            handle.write(json.dumps(payload, ensure_ascii=True))
            handle.write("\n")
            written += 1

    return written


def write_dataset_splits(
    train_path: Path = DEFAULT_TRAIN_PATH,
    valid_path: Path = DEFAULT_VALID_PATH,
    train_samples: int = 60_000,
    valid_samples: int = 2_000,
    seed: int = 42,
) -> dict[str, int | str]:
    generator = TerryDatasetGenerator(seed=seed)
    seen: set[str] = set()

    train_written = write_split(
        path=train_path,
        count=train_samples,
        generator=generator,
        split_name="train",
        seen=seen,
    )
    valid_written = write_split(
        path=valid_path,
        count=valid_samples,
        generator=generator,
        split_name="valid",
        seen=seen,
    )

    return {
        "train_path": str(train_path),
        "valid_path": str(valid_path),
        "train_samples": train_written,
        "valid_samples": valid_written,
    }

# 4. Model Components

Define core model components: RMSNorm, SelfAttention, SwiGLU, and TransformerBlock.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return self.weight * x * torch.rsqrt(norm + self.eps)


class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len=8192, sliding_window=2048, use_sliding_window=True):
        super().__init__()

        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.max_seq_len = max_seq_len
        self.sliding_window = sliding_window
        self.use_sliding_window = use_sliding_window

        self.qkv = nn.Linear(d_model, d_model * 3, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

        self.register_buffer("cos", None, persistent=False)
        self.register_buffer("sin", None, persistent=False)
        self.register_buffer("causal_mask", None, persistent=False)

    def _build_sliding_window_mask(self, max_len, device):
        """Build sliding window attention mask with local window + global attention."""
        causal_mask = torch.tril(torch.ones(max_len, max_len, device=device)).bool()
        
        if not self.use_sliding_window:
            return causal_mask
            
        window_mask = torch.zeros(max_len, max_len, device=device, dtype=torch.bool)
        
        for i in range(max_len):
            start = max(0, i - self.sliding_window + 1)
            window_mask[i, start:i+1] = True
            
            global_step = self.sliding_window // 4
            for j in range(0, i+1, global_step):
                window_mask[i, j] = True
        
        return causal_mask & window_mask

    def forward(self, x, padding_mask=None):
        """
        Args:
            x: Tensor of shape (B, T, C).
            padding_mask: Optional boolean tensor of shape (B, T).
                ``True`` marks valid tokens and ``False`` marks padding.
        """
        batch_size, seq_len, channels = x.size()
        device = x.device

        if self.cos is None or self.cos.device != device:
            self.cos, self.sin = build_rope_cache(
                self.max_seq_len,
                self.head_dim,
                device,
            )

        if self.causal_mask is None or self.causal_mask.device != device:
            if self.use_sliding_window:
                self.causal_mask = self._build_sliding_window_mask(self.max_seq_len, device)
            else:
                self.causal_mask = torch.tril(
                    torch.ones(self.max_seq_len, self.max_seq_len, device=device)
                ).bool()

        causal_mask = self.causal_mask[:seq_len, :seq_len]

        qkv = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [
            tensor.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
            for tensor in qkv
        ]

        q = apply_rope(q, self.cos[:seq_len], self.sin[:seq_len])
        k = apply_rope(k, self.cos[:seq_len], self.sin[:seq_len])

        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = attn.masked_fill(
            ~causal_mask.unsqueeze(0).unsqueeze(0),
            float("-inf"),
        )

        if padding_mask is not None:
            key_padding_mask = padding_mask.unsqueeze(1).unsqueeze(2)
            query_padding_mask = padding_mask.unsqueeze(1).unsqueeze(-1)
            attn = attn.masked_fill(~key_padding_mask, float("-inf"))
            attn = attn.masked_fill(~query_padding_mask, float("-inf"))

        attn = torch.softmax(attn, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, channels)

        if padding_mask is not None:
            out = out * padding_mask.unsqueeze(-1)

        return self.out(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = RMSNorm(config.d_model)
        self.attn = SelfAttention(
            config.d_model,
            config.n_heads,
            config.max_seq_len,
            config.sliding_window,
            config.use_sliding_window,
        )
        self.norm2 = RMSNorm(config.d_model)
        self.ffn = SwiGLU(
            config.d_model,
            config.d_model * config.ffn_mult,
        )
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(self, x, padding_mask=None):
        """
        Args:
            x: (B, T, C)
            padding_mask: (B, T) boolean, True = valid token
        """
        attn_out = self.attn(self.norm1(x), padding_mask=padding_mask)
        x = x + self.dropout(attn_out)

        ffn_out = self.ffn(self.norm2(x))
        x = x + self.dropout(ffn_out)
        return x


class TinyLLM(nn.Module):
    def __init__(self, config, vocab_size: int = VOCAB_SIZE):
        super().__init__()
        self.config = config
        self.config.vocab_size = vocab_size

        self.embed = nn.Embedding(self.config.vocab_size, config.d_model)

        self.blocks = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config.n_layers)]
        )

        self.norm = RMSNorm(config.d_model)
        self.head = nn.Linear(config.d_model, self.config.vocab_size, bias=False)

        # Weight tying
        self.head.weight = self.embed.weight
        
        self.gradient_checkpointing = False

    def gradient_checkpointing_enable(self):
        self.gradient_checkpointing = True
        
    def gradient_checkpointing_disable(self):
        self.gradient_checkpointing = False

    def resize_token_embeddings(self, new_vocab_size: int):
        self.config.vocab_size = new_vocab_size

        new_embed = nn.Embedding(self.config.vocab_size, self.config.d_model)
        new_head = nn.Linear(self.config.d_model, self.config.vocab_size, bias=False)

        n = min(self.embed.weight.shape[0], new_embed.weight.shape[0])
        new_embed.weight.data[:n] = self.embed.weight.data[:n]
        new_head.weight.data[:n] = self.head.weight.data[:n]

        self.embed = new_embed
        self.head = new_head

        self.head.weight = self.embed.weight

    def forward(self, x, padding_mask=None):
        """
        Args:
            x: (B, T)
            padding_mask: (B, T) boolean
        """
        x = self.embed(x)
        
        if self.gradient_checkpointing and self.training:
            import torch.utils.checkpoint as checkpoint
            
            def create_custom_forward(module):
                def custom_forward(*inputs):
                    return module(*inputs)
                return custom_forward
            
            for block in self.blocks:
                x = checkpoint.checkpoint(create_custom_forward(block), x, padding_mask)
        else:
            for block in self.blocks:
                x = block(x, padding_mask=padding_mask)
                
        x = self.norm(x)
        return self.head(x)

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_length=100,
        temperature=1.0,
        top_k=50,
        top_p=0.9,
        do_sample=True,
        pad_token_id=None,
        eos_token_id=None,
    ):
        if pad_token_id is None:
            raise ValueError("pad_token_id must be provided")
        if eos_token_id is None:
            raise ValueError("eos_token_id must be provided")
        if max_length <= 0:
            raise ValueError("max_length must be positive")

        device = input_ids.device
        batch_size, seq_len = input_ids.shape
        temperature = max(temperature, 1e-5)

        if seq_len >= max_length:
            return input_ids[:, -max_length:].clone()

        generated = torch.full(
            (batch_size, max_length),
            pad_token_id,
            dtype=torch.long,
            device=device,
        )
        generated[:, :seq_len] = input_ids
        finished_sequences = torch.zeros(
            batch_size, dtype=torch.bool, device=device
        )

        for cur_len in range(seq_len, max_length):
            start_idx = max(0, cur_len - self.config.max_seq_len)
            input_slice = generated[:, start_idx:cur_len]

            padding_mask = input_slice != pad_token_id

            logits = self(
                input_slice,
                padding_mask=padding_mask,
            )
            next_token_logits = logits[:, -1, :] / temperature

            if do_sample:
                if top_p < 1.0:
                    next_token_logits = self._top_p_filter(next_token_logits, top_p)

                if top_k > 0:
                    next_token_logits = self._top_k_filter(next_token_logits, top_k)

                probs = F.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = torch.argmax(
                    next_token_logits, dim=-1, keepdim=True
                )

            next_token[finished_sequences] = pad_token_id
            generated[:, cur_len] = next_token.squeeze(-1)

            finished_sequences |= next_token.squeeze(-1) == eos_token_id

            if torch.all(finished_sequences):
                break

        return generated

    def _top_p_filter(self, logits, top_p):
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        sorted_indices_to_remove = cum_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1]
        sorted_indices_to_remove[..., 0] = False
        
        sorted_logits[sorted_indices_to_remove] = float("-inf")
        
        unsorted_logits = torch.zeros_like(sorted_logits)
        unsorted_logits.scatter_(
            dim=-1,
            index=sorted_indices,
            src=sorted_logits,
        )
        return unsorted_logits

    def _top_k_filter(self, logits, top_k):
        top_k_logits, _ = torch.topk(logits, top_k, dim=-1)
        min_top_k = top_k_logits[..., -1:]
        return torch.where(logits < min_top_k, torch.tensor(float("-inf"), device=logits.device), logits)

# 5. Training & Evaluation

Training loop, validation, and model evaluation utilities.

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, scaler, grad_accum, log_interval=100):
    model.train()
    total_loss = 0.0
    num_batches = 0
    micro_step = 0

    for batch_idx, batch in enumerate(dataloader):
        x, y = batch
        x, y = x.to(device), y.to(device)
        
        # Create padding mask
        padding_mask = x != PAD_TOKEN_ID

        if scaler is not None:
            # Mixed precision training
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
                logits = model(x, padding_mask=padding_mask)
                
                # Reshape for cross entropy
                logits_flat = logits[:, :-1, :].contiguous().view(-1, logits.size(-1))
                y_flat = y[:, 1:].contiguous().view(-1)
                
                loss = F.cross_entropy(logits_flat, y_flat, ignore_index=PAD_TOKEN_ID)

            scaler.scale(loss).backward()
        else:
            logits = model(x, padding_mask=padding_mask)
            
            logits_flat = logits[:, :-1, :].contiguous().view(-1, logits.size(-1))
            y_flat = y[:, 1:].contiguous().view(-1)
            
            loss = F.cross_entropy(logits_flat, y_flat, ignore_index=PAD_TOKEN_ID)
            loss.backward()

        micro_step += 1
        
        if micro_step % grad_accum == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item()
        num_batches += 1

        if batch_idx % log_interval == 0:
            current_lr = scheduler.get_last_lr()[0] if scheduler.get_last_lr() else 0.0
            avg_loss = total_loss / max(1, num_batches)
            print(f'Batch {batch_idx:4d} | Loss: {loss.item():.4f} | Avg Loss: {avg_loss:.4f} | LR: {current_lr:.2e}')

    avg_loss = total_loss / num_batches if num_batches > 0 else float('inf')
    return avg_loss


@torch.no_grad()
def validate_epoch(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    for batch in dataloader:
        x, y = batch
        x, y = x.to(device), y.to(device)
        
        padding_mask = x != PAD_TOKEN_ID
        logits = model(x, padding_mask=padding_mask)
        
        logits_flat = logits[:, :-1, :].contiguous().view(-1, logits.size(-1))
        y_flat = y[:, 1:].contiguous().view(-1)
        
        loss = F.cross_entropy(logits_flat, y_flat, ignore_index=PAD_TOKEN_ID)
        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches if num_batches > 0 else float('inf')
    return avg_loss


def save_checkpoint(model, optimizer, scheduler, scaler, epoch, loss, checkpoint_path):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss,
        'config': model.config,
    }
    if scaler is not None:
        checkpoint['scaler_state_dict'] = scaler.state_dict()
    
    torch.save(checkpoint, checkpoint_path)


def load_checkpoint(checkpoint_path, model, optimizer, scheduler, scaler, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if scaler is not None and 'scaler_state_dict' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    return epoch, loss


def train_model(
    model,
    train_dataloader,
    valid_dataloader,
    optimizer,
    scheduler,
    scaler,
    device,
    train_cfg,
    num_epochs=2,
):
    best_loss = float('inf')
    patience_counter = 0
    patience = 3

    for epoch in range(num_epochs):
        print(f'\n=== Epoch {epoch + 1}/{num_epochs} ===')

        train_loss = train_epoch(
            model=model,
            dataloader=train_dataloader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            scaler=scaler,
            grad_accum=train_cfg.grad_accum,
        )

        valid_loss = validate_epoch(model, valid_dataloader, device)

        print(f'Epoch {epoch + 1:2d} | Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f}')

        if valid_loss < best_loss:
            best_loss = valid_loss
            patience_counter = 0
            save_checkpoint(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                loss=valid_loss,
                checkpoint_path='checkpoints/best_model.pt',
            )
            print(f'Saved checkpoint with validation loss: {valid_loss:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch + 1}')
                break

    print('Training completed')

# 6. Inference & Generation

Text generation and model evaluation utilities.

In [ ]:
@torch.no_grad()
def generate_text(
    model: TinyLLM,
    tokenizer: ByteTokenizer,
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.9,
    device: torch.device = torch.device('cpu'),
) -> str:
    model.eval()
    input_ids = tokenizer.encode(prompt, add_special_tokens=True)
    input_ids = torch.tensor([input_ids], dtype=torch.long, device=device)

    generated = model.generate(
        input_ids=input_ids,
        max_length=input_ids.shape[1] + max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_text = tokenizer.decode(generated[0].tolist(), skip_special_tokens=True)
    return generated_text


def evaluate_perplexity(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in dataloader:
            x, y = batch
            x, y = x.to(device), y.to(device)
            
            padding_mask = x != PAD_TOKEN_ID
            logits = model(x, padding_mask=padding_mask)
            
            logits_flat = logits[:, :-1, :].contiguous().view(-1, logits.size(-1))
            y_flat = y[:, 1:].contiguous().view(-1)
            
            loss = F.cross_entropy(logits_flat, y_flat, ignore_index=PAD_TOKEN_ID)

            if loss is not None:
                total_loss += loss.item() * y.numel()
                total_tokens += y.numel()

    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    perplexity = math.exp(avg_loss) if avg_loss < 100 else float('inf')
    return perplexity


def chat_with_terry(model, tokenizer, device):
    print("Chat with Terry! Type 'quit' to exit.")
    print("-" * 50)

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        if not user_input:
            continue

        response = generate_text(
            model=model,
            tokenizer=tokenizer,
            prompt=user_input,
            max_new_tokens=150,
            temperature=0.8,
            top_k=40,
            top_p=0.9,
            device=device,
        )

        print(f"Terry: {response}")
        print("-" * 50)


def load_trained_model(checkpoint_path: str | Path, device: torch.device) -> TinyLLM:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config = checkpoint['config']
    model = TinyLLM(config).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model


def run_inference_demo(model, tokenizer, device):
    test_prompts = [
        "hi terry, how are you today",
        "what do you see in the kitchen",
        "tell me a tiny story",
        "do you like the color blue",
        "what should i do with this spoon",
    ]

    print("Inference Demo:")
    print("=" * 50)

    for prompt in test_prompts:
        response = generate_text(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=100,
            temperature=0.7,
            device=device,
        )

        print(f"Prompt: {prompt}")
        print(f"Response: {response}")
        print("-" * 50)

# 7. Setup & Utilities

Device setup and data preparation utilities.

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def setup_device(device_cfg: str):
    if device_cfg == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device_cfg)

    print(f"Using device: {device}")

    if device.type == "cuda":
        torch.set_float32_matmul_precision("high")
    else:
        torch.set_num_threads(1)

    return device


def build_dataloader(tokenizer, model_cfg, train_cfg, use_cuda):
    """Build the Terry training dataloader."""
    train_tokens = Path(train_cfg.train_tokens_path)
    
    # Create a simple iterable dataset for token sequences
    class SimpleTokenDataset(IterableDataset):
        def __init__(self, path, max_seq_len, min_seq_len=32):
            self.path = Path(path)
            self.max_seq_len = max_seq_len
            self.min_seq_len = min_seq_len
            
        def __iter__(self):
            with self.path.open('r') as f:
                for line in f:
                    tokens = list(map(int, line.strip().split()))
                    if len(tokens) < self.min_seq_len:
                        continue
                    
                    # Chunk into sequences
                    for i in range(0, len(tokens) - 1, self.max_seq_len // 2):
                        seq = tokens[i:i + self.max_seq_len + 1]
                        if len(seq) > self.min_seq_len:
                            input_ids = torch.tensor(seq[:-1], dtype=torch.long)
                            target_ids = torch.tensor(seq[1:], dtype=torch.long)
                            yield input_ids, target_ids
    
    dataset = SimpleTokenDataset(
        path=train_cfg.train_tokens_path,
        max_seq_len=model_cfg.max_seq_len,
        min_seq_len=32,
    )

    loader = DataLoader(
        dataset,
        batch_size=train_cfg.batch_size,
        pin_memory=use_cuda,
        num_workers=0,
        collate_fn=build_collate_fn(tokenizer),
    )

    return loader


def build_valid_dataloader(tokenizer, model_cfg, train_cfg, use_cuda):
    """Build the validation dataloader."""
    class SimpleTokenDataset(IterableDataset):
        def __init__(self, path, max_seq_len, min_seq_len=32):
            self.path = Path(path)
            self.max_seq_len = max_seq_len
            self.min_seq_len = min_seq_len
            
        def __iter__(self):
            with self.path.open('r') as f:
                for line in f:
                    tokens = list(map(int, line.strip().split()))
                    if len(tokens) < self.min_seq_len:
                        continue
                    
                    for i in range(0, len(tokens) - 1, self.max_seq_len // 2):
                        seq = tokens[i:i + self.max_seq_len + 1]
                        if len(seq) > self.min_seq_len:
                            input_ids = torch.tensor(seq[:-1], dtype=torch.long)
                            target_ids = torch.tensor(seq[1:], dtype=torch.long)
                            yield input_ids, target_ids
    
    dataset = SimpleTokenDataset(
        path=train_cfg.valid_tokens_path,
        max_seq_len=model_cfg.max_seq_len,
        min_seq_len=32,
    )

    loader = DataLoader(
        dataset,
        batch_size=train_cfg.batch_size,
        pin_memory=use_cuda,
        num_workers=0,
        collate_fn=build_collate_fn(tokenizer),
    )

    return loader


def setup_training_components(model_cfg, train_cfg, use_cuda):
    device = setup_device(train_cfg.device)

    model = TinyLLM(model_cfg, vocab_size=VOCAB_SIZE).to(device)
    
    # Enable gradient checkpointing for memory efficiency with long sequences
    if model_cfg.max_seq_len > 4096:
        model.gradient_checkpointing_enable()
    
    # Mixed precision training
    scaler = None
    if train_cfg.use_mixed_precision and device.type == "cuda":
        scaler = torch.amp.GradScaler()
        print("Using mixed precision training")

    optimizer = AdamW(
        model.parameters(),
        lr=train_cfg.lr,
        weight_decay=train_cfg.weight_decay,
        fused=use_cuda and device.type == "cuda",
    )

    def lr_lambda(step: int):
        if step < train_cfg.warmup_steps:
            return step / max(1, train_cfg.warmup_steps)

        progress = (step - train_cfg.warmup_steps) / max(
            1, train_cfg.total_steps - train_cfg.warmup_steps
        )
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    return model, optimizer, scheduler, scaler, device

# 8. Execute Training

In [ ]:
# Main execution - Configuration
# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configuration
model_cfg = ModelConfig()
train_cfg = TrainConfig()

# Display configuration
print("=" * 60)
print("Model Configuration")
print("=" * 60)
print(f"d_model: {model_cfg.d_model}")
print(f"n_layers: {model_cfg.n_layers}")
print(f"n_heads: {model_cfg.n_heads}")
print(f"max_seq_len: {model_cfg.max_seq_len}")
print(f"sliding_window: {model_cfg.sliding_window}")
print(f"use_sliding_window: {model_cfg.use_sliding_window}")

print("\n" + "=" * 60)
print("Training Configuration")
print("=" * 60)
print(f"lr: {train_cfg.lr}")
print(f"batch_size: {train_cfg.batch_size}")
print(f"grad_accum: {train_cfg.grad_accum}")
print(f"max_seq_len: {model_cfg.max_seq_len}")
print(f"use_mixed_precision: {train_cfg.use_mixed_precision}")

# Check CUDA availability
use_cuda = torch.cuda.is_available()
logger.info(f'CUDA available: {use_cuda}')
if use_cuda:
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Set seed
set_seed(train_cfg.seed)

print("\n" + "=" * 60)
print("Ready to train!")
print("=" * 60)

# 9. Training Execution

Train the model on the dataset.

In [ ]:
# Prepare data (if needed)
tokenizer = load_tokenizer(train_cfg.tokenizer_dir)

# Check if token files exist
train_tokens_path = Path(train_cfg.train_tokens_path)
valid_tokens_path = Path(train_cfg.valid_tokens_path)

if not train_tokens_path.exists():
    print(f"Error: Training tokens not found at {train_tokens_path}")
    print("Please run prepare_data.py first or create the tokens files")
else:
    print(f"✓ Training tokens found: {train_tokens_path}")

if not valid_tokens_path.exists():
    print(f"Error: Validation tokens not found at {valid_tokens_path}")
    print("Please run prepare_data.py first or create the tokens files")
else:
    print(f"✓ Validation tokens found: {valid_tokens_path}")

print(f"\nTokenizer vocab size: {len(tokenizer)}")
print(f"Tokenizer special tokens:")
print(f"  pad_token_id: {tokenizer.pad_token_id}")
print(f"  bos_token_id: {tokenizer.bos_token_id}")
print(f"  eos_token_id: {tokenizer.eos_token_id}")

In [ ]:
# Build dataloaders and training components
print("\nBuilding dataloaders...")
train_dataloader = build_dataloader(tokenizer, model_cfg, train_cfg, use_cuda)
valid_dataloader = build_valid_dataloader(tokenizer, model_cfg, train_cfg, use_cuda)

print("Setting up training components...")
model, optimizer, scheduler, scaler, device = setup_training_components(
    model_cfg, train_cfg, use_cuda
)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Train the model
print("\nStarting training...")
train_model(
    model=model,
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    device=device,
    train_cfg=train_cfg,
    num_epochs=2,
)

# 10. Inference Demo

Generate text and test the trained model.

In [ ]:
# Run inference demo
print("\n" + "=" * 60)
print("Inference Demo")
print("=" * 60)

model.eval()
run_inference_demo(model, tokenizer, device)

# Test perplexity
print("\n" + "=" * 60)
print("Evaluating model perplexity...")
print("=" * 60)

# Create a small sample for quick evaluation
sample_size = min(10, sum(1 for _ in valid_dataloader))
print(f"Evaluating on sample of batches...")

final_perplexity = evaluate_perplexity(model, valid_dataloader, device)
print(f"Model Perplexity: {final_perplexity:.2f}")

print("\n" + "=" * 60)
print("Training and evaluation completed!")
print("=" * 60)